# Problem 1 (2 pts)

In real-world applications (such as neural network weight initialization or financial risk bounds), we often want variables that cluster around zero like a normal distribution, but strictly avoid extreme outliers. A **truncated normal distribution** $\mathcal{TN}(0, 1; -s, s)$ has a density proportional to $\exp(-x^2 / 2)$ on the interval $[-s, s]$ and zero elsewhere.

1. Implement a sampler using **naive rejection from a standard normal distribution** $\mathcal{N}(0, 1)$ (drawing from $\mathcal{N}(0, 1)$ and discarding values outside $[-s, s]$).
2. Implement a sampler using **rejection sampling with a uniform proposal** over $[-s, s]$. Determine the optimal scaling constant $M$ analytically.
3. Compare the empirical acceptance rates of both methods for $s = 0.1$ and $s = 3.0$. For which regimes of $s$ does each method dominate, and why?

In [3]:
# You're on your own, sorry!

# Problem 2 (2 pts)

## Markov Chain Monte Carlo (MCMC)

### Why move beyond rejection sampling?

Rejection sampling is conceptually elegant and exact: it is a *Las Vegas* algorithm that outputs independent samples with zero approximation error. However, it relies on finding a proposal density $g(x)$ and an envelope constant $M$ such that:

$$M g(x) \ge f(x) \quad \text{for all } x.$$

In practical scientific applications, this condition fails for two major reasons:

1. **The Curse of Dimensionality:** In $d$ dimensions, probability mass tends to concentrate in narrow, complex regions. The ratio of the target's volume to any simple bounding proposal's volume vanishes exponentially:

$$\mathbb{P}(\text{accept}) = \frac{1}{M} \sim \mathcal{O}(e^{-c \cdot d}).$$



For $d \ge 20$, the expected number of draws before a single acceptance can easily exceed the age of the universe.

2. **Intractable Geometry:** In Bayesian inference, posterior distributions $p(\theta \mid \mathcal{D}) \propto p(\mathcal{D} \mid \theta)p(\theta)$ often feature curved ridges and tight degeneracies. Constructing an analytical proposal $g(\theta)$ that tightly bounds such surfaces without knowing where the mass concentrates is practically impossible.

We need a method that does not attempt to enclose the entire distribution globally, but instead **navigates the high-probability landscape locally**.

---

### The core idea: Sampling along a trajectory

Instead of drawing independent points across the entire space and discarding the misses, MCMC constructs a **Markov chain** $\{X_0, X_1, X_2, \dots\}$ designed to have our target distribution $p(x) \propto f(x)$ as its unique **stationary distribution** (remember PageRank?).

* The sampler starts at an arbitrary point $X_0$.
* At each step $t$, it proposes a local perturbation $X^* \sim q(X^* \mid X_t)$.
* It decides whether to move to $X^*$ or stay at $X_t$ based on a local acceptance rule $\alpha(X_t, X^*)$.
* After a transient burn-in phase, the sequence of states $\{X_t\}$ mimics draws from $p(x)$: the chain naturally lingers in high-density regions and visits low-density regions rarely.

Unlike rejection sampling, an MCMC step that "rejects" does not throw away computation or restart a loop; it simply sets $X_{t+1} = X_t$, recording the current position again.

---

### Stationarity via detailed balance

Let $T(x \to y)$ be the transition kernel representing the probability (density) of stepping from state $x$ to state $y$.

A distribution $p(x)$ is **stationary** (invariant) under $T$ if running one more step leaves the distribution unchanged:

$$\int p(x) T(x \to y) \, dx = p(y).$$

Directly verifying this global balance integral is difficult. Instead, we enforce a much simpler local condition called **detailed balance** (microscopic reversibility):

$$p(x) T(x \to y) = p(y) T(y \to x) \quad \text{for all } x, y.$$

#### Proof that detailed balance implies stationarity

Integrating both sides with respect to $x$:

$$\int p(x) T(x \to y) \, dx = \int p(y) T(y \to x) \, dx = p(y) \underbrace{\int T(y \to x) \, dx}_{= 1} = p(y).$$

Detailed balance holds, so $p(x)$ is guaranteed to be a stationary distribution.

#### The Metropolis choice

If transitions are generated by proposing $y \sim q(y \mid x)$ and accepting with probability $\alpha(x, y)$, then for $x \neq y$:


$$T(x \to y) = q(y \mid x)\alpha(x, y).$$

To satisfy detailed balance, we set:


$$\alpha(x, y) = \min\left(1, \; \frac{p(y) q(x \mid y)}{p(x) q(y \mid x)}\right) = \min\left(1, \; \frac{f(y) q(x \mid y)}{f(x) q(y \mid x)}\right).$$

The normalizing constant of $p(x)$ cancels completely out of the ratio.

> **Ergodicity guarantee:** Detailed balance guarantees that $p(x)$ is *a* stationary distribution. If the chain is also **irreducible** (any state with $p(x) > 0$ can eventually be reached) and **aperiodic** (it does not cycle deterministically), the ergodic theorem guarantees that the distribution of $X_t$ converges uniquely to $p(x)$ from any starting point $X_0$.

---

### Concrete example: Sampling from a truncated normal

Consider sampling from a standard normal distribution constrained to the interval $[-s, s]$:

$$f(x) = \begin{cases} \exp\left(-\frac{x^2}{2}\right) & \text{if } x \in [-s, s], \\ 0 & \text{otherwise.} \end{cases}$$

#### Random-walk Metropolis formulation

1. **Current state:** $X_t \in [-s, s]$.
2. **Symmetric proposal:** Propose $X^* = X_t + \epsilon$, where $\epsilon \sim \mathcal{N}(0, \sigma_{\text{step}}^2)$. Because $q(X^* \mid X_t) = q(X_t \mid X^*)$, the proposal terms cancel:

$$\alpha(X_t, X^*) = \min\left(1, \; \frac{f(X^*)}{f(X_t)}\right).$$


3. **Acceptance evaluation:**
* If $\vert{}X^*\vert{} > s$, then $f(X^*) = 0 \implies \alpha = 0$ (**instant rejection**; boundary bounce).
* If $\vert{}X^*\vert{} \le s$, then:

$$\alpha = \min\left(1, \; \exp\left(-\frac{(X^*)^2 - X_t^2}{2}\right)\right).$$

* If $\alpha \ge u \sim \operatorname{Uniform}(0, 1)$, set $X_{t+1} = X^*$; otherwise, set $X_{t+1} = X_t$.

#### Autocorrelation function (ACF)

In standard Monte Carlo, every sample is drawn independently: knowing $X_t$ tells you nothing about $X_{t+1}$.

In MCMC, consecutive samples are intrinsically dependent: $X_{t+1}$ depends directly on $X_t$. The **Autocorrelation Function (ACF)** measures how strongly samples separated by a time lag $k$ remain correlated with one another.

For a stationary chain $\{X_1, X_2, \dots, X_N\}$ with sample mean $\bar{X}$ and sample variance $\hat{\sigma}^2$, the sample autocorrelation at **lag $k$** is:

$$\rho_k = \frac{\frac{1}{N - k} \sum_{t=1}^{N - k} (X_t - \bar{X})(X_{t + k} - \bar{X})}{\hat{\sigma}^2}$$

* $\rho_0 = 1.0$ (every sample is perfectly correlated with itself).
* As $k$ grows, $\rho_k$ typically decays toward $0$.
* **Fast decay:** The chain quickly "forgets" its past and explores space efficiently.
* **Slow decay:** The chain is sluggish, heavily autocorrelated, and moves like slow diffusion.

#### Why ACF matters: Effective sample size (ESS)

Because samples are correlated, $N = 10{,}000$ MCMC steps **do not provide $10{,}000$ independent pieces of information**.

The variance of our estimator $\hat{\mu} = \frac{1}{N} \sum_{t=1}^N X_t$ is inflated by the sum of autocorrelations:

$$\operatorname{Var}(\hat{\mu}) \approx \frac{\sigma^2}{N} \underbrace{\left(1 + 2 \sum_{k=1}^\infty \rho_k\right)}_{\text{Autocorrelation Time } \tau_{\text{int}}}$$

The quantity $\tau_{\text{int}} = 1 + 2 \sum_{k=1}^\infty \rho_k$ is the **integrated autocorrelation time**. It defines the **Effective Sample Size (ESS)**:

$$\text{ESS} = \frac{N}{\tau_{\text{int}}} = \frac{N}{1 + 2 \sum_{k=1}^\infty \rho_k}$$

* If $\tau_{\text{int}} = 20$, you need $20$ MCMC steps to get the statistical information equivalent to **$1$ independent draw**.
* If a run has high autocorrelation, an apparently huge sample size ($N = 100{,}000$) might yield an ESS of only a few hundred.

**Task:** Implement the MCMC algorithm above.
1. Run the truncated normal sampler using three distinct values of $\sigma_{\text{step}}$: too small (e.g., $0.02$), optimal ($0.5$), and too large ($5.0$). For each run:
   * Record the overall empirical acceptance rate.
   * Plot the first $1{,}000$ iterations as a trace plot (iteration $t$ vs. value $X_t$).
   * Plot the autocorrelation function (ACF) up to lag $k = 50$.
   * Explain how both extremes inflate autocorrelation and reduce the Effective Sample Size (ESS).
2. Study the lack of stationarity in burn-in samples:
   * Initialize the chain deliberately at the extreme boundary $X_0 = s$. Plot the cumulative running mean $\bar{X}_t = \frac{1}{t} \sum_{i=1}^t X_i$ for $t = 1, \dots, 5{,}000$. Contrast this with the cumulative mean computed after discarding the first $500$ steps.
   * Initialize 4 independent chains with starting points $X_0 \in \{-s, -s/2, s/2, s\}$. Overlay their trace plots for the first $300$ steps. Identify the transition point (iteration number) where the initial conditions are forgotten and the chains fully overlap.

In [4]:
# You're on your own, sorry!

# Problem 3 (3 pts)

Markov Chain Monte Carlo can also be used for sampling from discrete distributions, essentially representing a graph walk.

**Task**: Implement an MCMC scheme for uniformly sampling a random Sudoku board. The board should not be completed but must be valid. Pay attention to the balance equation!

In [5]:
# You're on your own, sorry!